## Explainable CNN Brain Tumor Classification

## 1. Mount Google Drive & Import Library


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import json
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU tersedia       : {tf.config.list_physical_devices("GPU")}')


## 2. Konfigurasi Path & Grid Hyperparameter


In [ ]:
# Path dataset
TRAIN_DIR      = '/content/drive/MyDrive/Tugas Akhir/FINAL_GC_test/train'
VAL_DIR        = '/content/drive/MyDrive/Tugas Akhir/FINAL_GC_test/val'
TEST_DIR       = '/content/drive/MyDrive/Tugas Akhir/FINAL_GC_test/test'
TRAIN_DIR_ORIG = TRAIN_DIR  # data asli (sebelum augmentasi), dipakai Cell 14 untuk CV

# Folder output
OUTPUT_DIR = '/content/drive/MyDrive/Tugas Akhir/FINAL_EXP_CNN_GC_test_TANPA_RESIZE'
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Hyperparameter tetap
IMG_SIZE    = (224, 224)
EPOCHS      = 40  # max epoch; EarlyStopping berhenti lebih cepat
NUM_CLASSES = 2
CLASS_NAMES = ['no_tumor', 'tumor']

# Grid hyperparameter
PARAM_GRID = {
    'learning_rate': [0.0001, 0.001, 0.01],
    'batch_size'   : [16, 40, 64],
    'dropout_rate' : [0.1, 0.2, 0.3],
}

REDUCED_GRID = False  # True = subset 7 kombinasi, False = full 27 kombinasi
FIXED_OPTIMIZER = 'adam'  # tidak ikut di-tune

# Subset kombinasi jika REDUCED_GRID=True (baseline + variasi 1 param)
REDUCED_COMBINATIONS = [
    (0.001,  40,   0.5),   # baseline
    (0.0001, 40,   0.5),
    (0.01,   40,   0.5),
    (0.001,  16,   0.5),
    (0.001,  64,   0.5),
    (0.001,  40,   0.3),
    (0.001,  40,   0.6),
]

print(f'Output: {OUTPUT_DIR} | Mode: {"REDUCED" if REDUCED_GRID else "FULL"} | Epochs: {EPOCHS}')


## 2.5 Augmentasi Data

In [ ]:
import cv2
import glob
import random
import shutil

# Path augmentasi
TRAIN_DIR_AUG = '/content/drive/MyDrive/Tugas Akhir/FINAL_EXP_CNN_GC_test_TANPA_RESIZE/dataset_augmented/train'
VAL_DIR_AUG   = VAL_DIR   # val & test tidak diaugmentasi
TEST_DIR_AUG  = TEST_DIR

TARGET_MULTIPLIER = 4  # total = 1 asli + 3 augmented
AUG_PER_METHOD     = TARGET_MULTIPLIER - 1
AUG_SEED = 42


def apply_shearing(img_rgb: np.ndarray, shear_range: tuple = (-15, 15), rng: random.Random = None) -> np.ndarray:
    """Shearing citra pada sumbu x & y, pusat di tengah agar lesi tidak bergeser."""
    if rng is None:
        rng = random.Random()
    shear_x_deg = rng.uniform(shear_range[0], shear_range[1])
    shear_y_deg = rng.uniform(shear_range[0], shear_range[1])
    sx = np.tan(np.deg2rad(shear_x_deg))
    sy = np.tan(np.deg2rad(shear_y_deg))
    h, w = img_rgb.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M = np.float32([
        [1,  sx, -cy * sx],
        [sy,  1, -cx * sy]
    ])
    return cv2.warpAffine(img_rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)


def apply_rotation(img_rgb: np.ndarray, angle_range: tuple = (-25, 25), rng: random.Random = None) -> np.ndarray:
    """Rotasi citra dengan sudut acak, berpusat di tengah citra."""
    if rng is None:
        rng = random.Random()
    angle = rng.uniform(angle_range[0], angle_range[1])
    h, w  = img_rgb.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle, scale=1.0)
    return cv2.warpAffine(img_rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)


def apply_translation(img_rgb: np.ndarray, tx_range: tuple = (-15, 15), ty_range: tuple = (-15, 15), rng: random.Random = None) -> np.ndarray:
    """Geser piksel citra pada sumbu x & y secara acak."""
    if rng is None:
        rng = random.Random()
    tx = rng.uniform(tx_range[0], tx_range[1])
    ty = rng.uniform(ty_range[0], ty_range[1])
    h, w = img_rgb.shape[:2]
    M = np.float32([[1, 0, tx],
                    [0, 1, ty]])
    return cv2.warpAffine(img_rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)


def apply_rotation_then_translation(img_rgb: np.ndarray, angle_range: tuple = (-25, 25), tx_range: tuple = (-15, 15), ty_range: tuple = (-15, 15), rng: random.Random = None) -> np.ndarray:
    img = apply_rotation(img_rgb, angle_range, rng)
    img = apply_translation(img, tx_range, ty_range, rng)
    return img


def apply_translation_then_shearing(img_rgb: np.ndarray, tx_range: tuple = (-15, 15), ty_range: tuple = (-15, 15), shear_range: tuple = (-15, 15), rng: random.Random = None) -> np.ndarray:
    img = apply_translation(img_rgb, tx_range, ty_range, rng)
    img = apply_shearing(img, shear_range, rng)
    return img


def apply_translation_shearing_rotation(img_rgb: np.ndarray, tx_range: tuple = (-15, 15), ty_range: tuple = (-15, 15), shear_range: tuple = (-15, 15), angle_range: tuple = (-25, 25), rng: random.Random = None) -> np.ndarray:
    """Pipeline: translasi -> shearing -> rotasi (metode augmentasi terkuat)."""
    img = apply_translation(img_rgb, tx_range, ty_range, rng)
    img = apply_shearing(img, shear_range, rng)
    img = apply_rotation(img, angle_range, rng)
    return img


# Registry metode augmentasi: (kode_suffix, fungsi, kwargs)
AUGMENTATION_METHODS = [
    ('shear',           apply_shearing,                      {}),
    ('rot',             apply_rotation,                      {}),
    ('trans_shear_rot', apply_translation_shearing_rotation, {}),
]


def run_augmentation(
    src_train_dir : str,
    dst_train_dir : str,
    class_names   : list,
    aug_per_method: int   = 2,
    img_size      : tuple = (224, 224),
    seed          : int   = 42,
    overwrite     : bool  = False,
) -> None:
    """Buat dataset training augmented di dst_train_dir (asli + salinan round-robin ke 3 metode)."""
    if os.path.isdir(dst_train_dir) and not overwrite:
        existing = glob.glob(os.path.join(dst_train_dir, '**', '*.*'), recursive=True)
        if existing:
            return

    rng = random.Random(seed)
    n_methods = len(AUGMENTATION_METHODS)

    for cls_name in class_names:
        src_cls = os.path.join(src_train_dir, cls_name)
        dst_cls = os.path.join(dst_train_dir, cls_name)
        os.makedirs(dst_cls, exist_ok=True)

        img_paths = sorted(
            p for ext in ('*.png', '*.jpg', '*.jpeg')
            for p in glob.glob(os.path.join(src_cls, ext))
        )
        if not img_paths:
            continue

        for img_path in img_paths:
            fname     = os.path.basename(img_path)
            name, ext = os.path.splitext(fname)

            raw = cv2.imread(img_path)
            if raw is None:
                continue
            img_rgb = cv2.cvtColor(cv2.resize(raw, img_size), cv2.COLOR_BGR2RGB)

            cv2.imwrite(os.path.join(dst_cls, fname), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

            # Distribusikan salinan augmented secara round-robin ke tiap metode
            aug_counter = 1
            for slot_idx in range(aug_per_method):
                method_suffix, aug_fn, kwargs = AUGMENTATION_METHODS[slot_idx % n_methods]
                aug_img   = aug_fn(img_rgb, rng=rng, **kwargs)
                aug_fname = f'{name}_aug{aug_counter:02d}_{method_suffix}_s{slot_idx + 1}{ext}'
                cv2.imwrite(os.path.join(dst_cls, aug_fname), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
                aug_counter += 1


TRAIN_DIR_ORIG = TRAIN_DIR
print(f'Augmentasi siap. TRAIN_DIR_ORIG={TRAIN_DIR_ORIG} | TRAIN_DIR_AUG={TRAIN_DIR_AUG}')


## 2.6 Augmentasi & Persiapan Data Training


In [ ]:
def load_train_rgb_array(train_dir, class_names, resize_to=None):
    """Load semua gambar training sebagai array RGB (float32, dinormalisasi 0-1)."""
    X_list, y_list = [], []
    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = os.path.join(train_dir, cls_name)
        if not os.path.isdir(cls_dir):
            continue
        paths_cls = sorted(
            p for ext in ('*.png', '*.jpg', '*.jpeg')
            for p in glob.glob(os.path.join(cls_dir, ext))
        )
        for fpath in paths_cls:
            raw = cv2.imread(fpath)
            if raw is None:
                continue
            img_rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            if resize_to:
                img_rgb = cv2.resize(img_rgb, resize_to, interpolation=cv2.INTER_LINEAR)
            img_norm = img_rgb.astype(np.float32) / 255.0
            X_list.append(img_norm)
            y_list.append(cls_idx)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)


# Langkah 1: load data asli
X_orig_arr, y_orig_arr = load_train_rgb_array(TRAIN_DIR_ORIG, CLASS_NAMES, resize_to=IMG_SIZE)

# Langkah 2: augmentasi (3 metode, round-robin -> 4x data asli)
run_augmentation(
    src_train_dir  = TRAIN_DIR_ORIG,
    dst_train_dir  = TRAIN_DIR_AUG,
    class_names    = CLASS_NAMES,
    aug_per_method = AUG_PER_METHOD,
    img_size       = IMG_SIZE,
    seed           = AUG_SEED,
    overwrite      = False,
)

# Langkah 3: load semua data (asli + augmented), lalu shuffle
X_train_final, y_train_final = load_train_rgb_array(TRAIN_DIR_AUG, CLASS_NAMES, resize_to=IMG_SIZE)
idx_shuf      = np.random.permutation(len(X_train_final))
X_train_final = X_train_final[idx_shuf]
y_train_final = y_train_final[idx_shuf]

# Alias untuk Grid Search
X_smote = X_train_final
y_smote = y_train_final

# Update TRAIN_DIR untuk generator val/test
TRAIN_DIR = TRAIN_DIR_AUG

print(f'Data training siap: {X_train_final.shape}, total={len(y_train_final)}')


## 3. Fungsi Utilitas


In [ ]:
def make_generators(batch_size):
    """Generator untuk val & test. Train pakai array X_smote/y_smote langsung."""
    val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='sparse', classes=CLASS_NAMES, shuffle=False
    )
    test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='sparse', classes=CLASS_NAMES, shuffle=False
    )
    return val_gen, test_gen


def build_cnn(dropout_rate=0.5, num_classes=2):
    """CNN Iftikhar et al. (2025). Filter: 8-16-32-64-128-256, Dense: 512-512."""
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='input_image')

    for filters, name in zip([8, 16, 32, 64, 128], ['conv1','conv2','conv3','conv4','conv5']):
        if name == 'conv1':
            x = layers.Conv2D(filters, (3,3), padding='same', activation='relu', name=name)(inputs)
        else:
            x = layers.Conv2D(filters, (3,3), padding='same', activation='relu', name=name)(x)
        x = layers.MaxPooling2D((2,2), name=f'pool{name[-1]}')(x)

    x = layers.Conv2D(256, (3,3), padding='same', activation='relu', name='conv6')(x)
    x = layers.BatchNormalization(name='batchnorm')(x)
    x = layers.AveragePooling2D((2,2), name='avgpool')(x)

    x = layers.Flatten(name='flatten')(x)
    x = layers.Dropout(dropout_rate, name='dropout1')(x)
    x = layers.Dense(512, activation='relu', name='dense1')(x)
    x = layers.Dropout(dropout_rate, name='dropout2')(x)
    x = layers.Dense(512, activation='relu', name='dense2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='CNN_Iftikhar2025')


def get_optimizer(name, lr):
    """Kembalikan instance optimizer berdasarkan nama string."""
    opts = {
        'adam'    : keras.optimizers.Adam(learning_rate=lr),
        'sgd'     : keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
        'rmsprop' : keras.optimizers.RMSprop(learning_rate=lr),
    }
    if name not in opts:
        raise ValueError(f'Optimizer tidak dikenal: {name}. Pilih: {list(opts.keys())}')
    return opts[name]


def combo_name(lr, bs, dr):
    return f'lr{lr}_bs{bs}_dr{dr}'

## 4. Generate Kombinasi Grid


In [ ]:
if REDUCED_GRID:
    combinations = REDUCED_COMBINATIONS
else:
    combinations = list(itertools.product(
        PARAM_GRID['learning_rate'],
        PARAM_GRID['batch_size'],
        PARAM_GRID['dropout_rate'],
    ))

total_runs = len(combinations)
df_grid = pd.DataFrame(combinations, columns=['learning_rate', 'batch_size', 'dropout_rate'])
df_grid.index = df_grid.index + 1
df_grid.index.name = 'Run'

print(f'Total kombinasi: {total_runs}')


## 5. Grid Search Loop


In [ ]:
RESULTS_JSON = os.path.join(OUTPUT_DIR, 'grid_results.json')

# Load hasil sebelumnya jika ada (resume jika training terputus)
if os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON, 'r') as f:
        all_results = json.load(f)
    done_names = {r['name'] for r in all_results}
else:
    all_results = []
    done_names  = set()

print(f'Memulai {total_runs} run ({len(done_names)} sudah selesai sebelumnya)')


In [ ]:
xgrid_start = time.time()

for run_idx, (lr, bs, dr) in enumerate(combinations, start=1):
    name = combo_name(lr, bs, dr)

    if name in done_names:  # resume: skip run yang sudah selesai
        continue

    run_start = time.time()

    val_gen, test_gen = make_generators(bs)  # train pakai array X_smote/y_smote

    cw_array = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_smote)
    class_weight_dict = dict(enumerate(cw_array))

    tf.keras.backend.clear_session()  # bebaskan memori GPU dari run sebelumnya
    model = build_cnn(dropout_rate=dr, num_classes=NUM_CLASSES)
    model.compile(
        optimizer=get_optimizer(FIXED_OPTIMIZER, lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    ckpt_path = os.path.join(MODELS_DIR, f'{name}.keras')
    callbacks = [
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss', save_best_only=True, mode='min', verbose=0),
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=0),
    ]

    history = model.fit(
        X_smote, y_smote,
        epochs=EPOCHS,
        batch_size=bs,
        validation_data=val_gen,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=0
    )

    epochs_run    = len(history.history['val_loss'])
    best_val_loss = min(history.history['val_loss'])
    best_val_acc  = max(history.history['val_accuracy'])

    test_gen.reset()
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)

    test_gen.reset()
    y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
    y_true = test_gen.classes

    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    f1_weighted = report['weighted avg']['f1-score']
    precision_w = report['weighted avg']['precision']
    recall_w    = report['weighted avg']['recall']

    # Specificity biner = TN / (TN + FP); kelas negatif = 'no_tumor' (index 0)
    cm_run = confusion_matrix(y_true, y_pred)
    tn_run, fp_run = cm_run[0, 0], cm_run[0, 1]
    specificity = tn_run / (tn_run + fp_run) if (tn_run + fp_run) > 0 else 0.0

    run_time = time.time() - run_start

    result = {
        'run'           : run_idx,
        'name'          : name,
        'learning_rate' : lr,
        'batch_size'    : bs,
        'dropout_rate'  : dr,
        'epochs_run'    : epochs_run,
        'best_val_loss' : round(best_val_loss, 6),
        'best_val_acc'  : round(best_val_acc,  6),
        'test_loss'     : round(test_loss, 6),
        'test_acc'      : round(test_acc,  6),
        'f1_weighted'   : round(f1_weighted, 6),
        'precision_w'   : round(precision_w,  6),
        'recall_w'      : round(recall_w,     6),
        'specificity'   : round(specificity,  6),
        'optimizer'     : FIXED_OPTIMIZER,
        'run_time_sec'  : round(run_time, 1),
        'history_val_loss': history.history['val_loss'],
        'history_val_acc' : history.history['val_accuracy'],
        'history_loss'    : history.history['loss'],
        'history_acc'     : history.history['accuracy'],
    }

    all_results.append(result)
    done_names.add(name)

    with open(RESULTS_JSON, 'w') as f:  # auto-save tiap run, tahan crash
        json.dump(all_results, f, indent=2)

    print(f'[{run_idx:02d}/{total_runs}] {name}: test_acc={test_acc:.4f} f1={f1_weighted:.4f}')


total_time = (time.time() - xgrid_start) / 60
print(f'Grid Search selesai. Total waktu: {total_time:.1f} menit')


## 6. Ringkasan Hasil


In [ ]:
with open(RESULTS_JSON, 'r') as f:
    all_results = json.load(f)

cols = ['run', 'learning_rate', 'batch_size', 'optimizer', 'dropout_rate',
        'epochs_run', 'best_val_acc', 'test_acc', 'f1_weighted',
        'precision_w', 'recall_w', 'specificity',
        'best_val_loss', 'test_loss', 'run_time_sec']

df_results = pd.DataFrame([{c: r.get(c, float('nan')) for c in cols} for r in all_results])

# Composite score: kombinasi 5 metrik, bobot Recall tertinggi karena
# False Negative (tumor tak terdeteksi) lebih berbahaya secara klinis
W_ACCURACY, W_PRECISION, W_RECALL, W_SPECIFICITY, W_F1 = 0.20, 0.15, 0.30, 0.15, 0.20
assert abs(W_ACCURACY + W_PRECISION + W_RECALL + W_SPECIFICITY + W_F1 - 1.0) < 1e-9

df_results['composite_score'] = (
    W_ACCURACY    * df_results['test_acc']     +
    W_PRECISION   * df_results['precision_w']  +
    W_RECALL      * df_results['recall_w']     +
    W_SPECIFICITY * df_results['specificity']  +
    W_F1          * df_results['f1_weighted']
)

df_results = df_results.sort_values('composite_score', ascending=False).reset_index(drop=True)

df_results['test_acc_%']      = (df_results['test_acc']        * 100).round(2)
df_results['best_val_acc_%']  = (df_results['best_val_acc']    * 100).round(2)
df_results['f1_%']            = (df_results['f1_weighted']     * 100).round(2)
df_results['precision_%']     = (df_results['precision_w']     * 100).round(2)
df_results['recall_%']        = (df_results['recall_w']        * 100).round(2)
df_results['specificity_%']   = (df_results['specificity']     * 100).round(2)
df_results['composite_%']     = (df_results['composite_score'] * 100).round(2)
df_results['time_min']        = (df_results['run_time_sec']    / 60).round(1)

display_cols_full = [
    'run', 'learning_rate', 'batch_size', 'dropout_rate',
    'test_acc_%', 'precision_%', 'recall_%', 'specificity_%', 'f1_%',
    'composite_%', 'time_min'
]

display(df_results[display_cols_full].rename(columns={
    'learning_rate' : 'LR',
    'batch_size'    : 'BS',
    'dropout_rate'  : 'Dropout',
    'test_acc_%'    : 'Accuracy (%)',
    'precision_%'   : 'Precision (%)',
    'recall_%'      : 'Recall (%)',
    'specificity_%' : 'Specificity (%)',
    'f1_%'          : 'F1 (%)',
    'composite_%'   : 'Composite Score (%)',
    'time_min'      : 'Waktu (mnt)',
}))

best = df_results.iloc[0]
print(f'Terbaik: lr={best.learning_rate} bs={int(best.batch_size)} dropout={best.dropout_rate} '
      f'| acc={best["test_acc_%"]:.2f}% f1={best["f1_%"]:.2f}% composite={best["composite_%"]:.2f}%')

best_by_acc = df_results.sort_values('test_acc', ascending=False).iloc[0]
if best_by_acc['run'] != best['run']:
    print(f'Catatan: model accuracy tertinggi berbeda dari composite terbaik '
          f'(lr={best_by_acc.learning_rate}, bs={int(best_by_acc.batch_size)}, dropout={best_by_acc.dropout_rate}, '
          f'acc={best_by_acc["test_acc_%"]:.2f}%) — composite dipilih karena lebih baik di Recall/Specificity.')


## 7. Simpan Tabel ke CSV


In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'grid_results_summary.csv')
df_results[display_cols_full].to_csv(csv_path, index=False)
print(f'Tabel disimpan ke: {csv_path}')

## 8. Test Accuracy per Kombinasi


In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(df_results)*0.8), 5))

labels = [f"lr={r['learning_rate']}\nbs={r['batch_size']}\ndr={r['dropout_rate']}"
          for _, r in df_results.iterrows()]

colors = ['#2ecc71' if i == 0 else '#3498db' if i < 3 else '#95a5a6'
          for i in range(len(df_results))]

bars = ax.bar(range(len(df_results)), df_results['test_acc_%'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, df_results['test_acc_%']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(range(len(df_results)))
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Grid Search — Test Accuracy per Kombinasi Hyperparameter\nHijau=Terbaik  Biru=Top-3',
             fontweight='bold')
ax.set_ylim(max(0, df_results['test_acc_%'].min() - 5), min(101, df_results['test_acc_%'].max() + 3))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'grid_bar_test_acc.png'), dpi=150)
plt.show()

## 9. Pengaruh Tiap Hyperparameter


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Pengaruh Tiap Hyperparameter terhadap Test Accuracy',
             fontsize=14, fontweight='bold')

params_to_plot = [
    ('learning_rate', 'Learning Rate', axes[0]),
    ('batch_size',    'Batch Size',    axes[1]),
    ('dropout_rate',  'Dropout Rate',  axes[2]),
]

for param, label, ax in params_to_plot:
    grouped = df_results.groupby(param)['test_acc_%']
    means = grouped.mean()
    stds  = grouped.std().fillna(0)

    x_vals = [str(v) for v in means.index]
    bars = ax.bar(x_vals, means.values,
                  yerr=stds.values, capsize=5,
                  color='#3498db', alpha=0.8, edgecolor='white')

    for xi, x_val in enumerate(means.index):
        subset_vals = df_results[df_results[param] == x_val]['test_acc_%']
        ax.scatter([xi] * len(subset_vals), subset_vals,
                   color='navy', zorder=5, s=30, alpha=0.7)

    ax.set_xlabel(label)
    ax.set_ylabel('Test Accuracy (%) — rata-rata ± std')
    ax.set_title(f'Rata-rata Test Acc by {label}', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for bar, mean_val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{mean_val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'grid_param_effect.png'), dpi=150)
plt.show()

## 10. Learning Rate vs Batch Size (Avg Test Acc)


In [ ]:
# Pivot: avg test accuracy across dropout untuk tiap (LR, BS)
pivot = df_results.pivot_table(
    values='test_acc_%',
    index='learning_rate',
    columns='batch_size',
    aggfunc='mean'
)

if pivot.shape[0] >= 2 and pivot.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        pivot, annot=True, fmt='.1f', cmap='YlGnBu',
        linewidths=0.5, cbar_kws={'label': 'Test Accuracy (%)'},
        ax=ax
    )
    ax.set_title('Heatmap: Avg Test Acc (LR × Batch Size)\n(dirata-ratakan atas Dropout Rate)',
                 fontweight='bold')
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Learning Rate')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'grid_heatmap_lr_bs.png'), dpi=150)
    plt.show()
else:
    print('Heatmap butuh >=2 nilai unik untuk LR dan Batch Size.')

## 11. Learning Curves Top-3 Model

In [ ]:
top3_names = df_results.head(3)['run'].tolist()
top3_results = [r for r in all_results if r['run'] in top3_names]
name_order = {r.run: i for i, r in enumerate(df_results.head(3).itertuples())}
top3_results.sort(key=lambda r: name_order.get(r['run'], 99))

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Learning Curves — Top-3 Kombinasi Terbaik', fontsize=14, fontweight='bold')

for row, r in enumerate(top3_results):
    epochs_x = range(1, len(r['history_val_loss']) + 1)
    title    = (f'#{row+1}: lr={r["learning_rate"]} | bs={r["batch_size"]} | '
                f'dr={r["dropout_rate"]}\n'
                f'Test Acc: {r["test_acc"]*100:.2f}% | F1: {r["f1_weighted"]*100:.2f}%')

    axes[row, 0].plot(epochs_x, r['history_acc'],     label='Train', color='royalblue')
    axes[row, 0].plot(epochs_x, r['history_val_acc'], label='Val',   color='tomato', linestyle='--')
    axes[row, 0].set_title(title, fontsize=9, fontweight='bold')
    axes[row, 0].set_ylabel('Accuracy')
    axes[row, 0].legend(fontsize=8)
    axes[row, 0].grid(alpha=0.3)

    axes[row, 1].plot(epochs_x, r['history_loss'],     label='Train', color='royalblue')
    axes[row, 1].plot(epochs_x, r['history_val_loss'], label='Val',   color='tomato', linestyle='--')
    axes[row, 1].set_ylabel('Loss')
    axes[row, 1].legend(fontsize=8)
    axes[row, 1].grid(alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'grid_top3_curves.png'), dpi=150)
plt.show()

## 12. Confusion Matrix Kombinasi Terbaik


In [ ]:
best_row = df_results.iloc[0]
best_lr  = best_row['learning_rate']
best_bs  = int(best_row['batch_size'])
best_dr  = best_row['dropout_rate']

best_ckpt = os.path.join(MODELS_DIR, f'{combo_name(best_lr, best_bs, best_dr)}.keras')
tf.keras.backend.clear_session()
best_model = keras.models.load_model(best_ckpt)

_, test_gen = make_generators(best_bs)
test_gen.reset()
y_pred = np.argmax(best_model.predict(test_gen, verbose=0), axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(
    f'Confusion Matrix — Kombinasi Terbaik\n'
    f'lr={best_lr} | bs={best_bs} | dropout={best_dr}',
    fontweight='bold'
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'grid_best_confusion_matrix.png'), dpi=150)
plt.show()


## 13. Simpan Model Terbaik


In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'best_model_grid_search.keras')
best_model.save(final_path)

best_config = {
    'learning_rate'  : best_lr,
    'batch_size'     : best_bs,
    'optimizer'      : FIXED_OPTIMIZER,
    'dropout_rate'   : best_dr,
    'test_acc'       : float(best_row['test_acc']),
    'f1_weighted'    : float(best_row['f1_weighted']),
    'precision_w'    : float(best_row['precision_w']),
    'recall_w'       : float(best_row['recall_w']),
    'specificity'    : float(best_row['specificity']),
    'composite_score': float(best_row['composite_score']),
}
with open(os.path.join(OUTPUT_DIR, 'best_config.json'), 'w') as f:
    json.dump(best_config, f, indent=2)

print(f'Model tersimpan: {final_path}')
print(json.dumps(best_config, indent=2))


## Ringkasan Grid Search


In [ ]:
# 14. Select-Shuffle-Test — Estimasi Performa Generalisasi

import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# Load konfigurasi terbaik dari Grid Search (fase SELECT)
best_config_path = os.path.join(OUTPUT_DIR, 'best_config.json')
with open(best_config_path, 'r') as f:
    best_config = json.load(f)

BEST_LR  = best_config['learning_rate']
BEST_BS  = int(best_config['batch_size'])
BEST_OPT = best_config['optimizer']
BEST_DR  = best_config['dropout_rate']


def collect_all_images(dirs, class_names):
    # Pakai TRAIN_DIR_ORIG (bukan TRAIN_DIR_AUG) agar data augmented tidak bocor ke CV
    all_paths, all_labels = [], []
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    for d in dirs:
        for cls_name, cls_idx in class_to_idx.items():
            cls_dir = os.path.join(d, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            for ext in ('*.png', '*.jpg', '*.jpeg'):
                for fpath in glob.glob(os.path.join(cls_dir, ext)):
                    if os.path.exists(fpath):
                        all_paths.append(fpath)
                        all_labels.append(cls_idx)
    return all_paths, np.array(all_labels)


all_paths, all_labels = collect_all_images(
    dirs=[TRAIN_DIR_ORIG, VAL_DIR, TEST_DIR],
    class_names=CLASS_NAMES
)

from tensorflow.keras.preprocessing import image as keras_image

def load_images_as_array(paths, img_size=(224, 224)):
    X = np.zeros((len(paths), *img_size, 3), dtype=np.float32)
    for i, p in enumerate(paths):
        try:
            img = keras_image.load_img(p, target_size=img_size)
            X[i] = keras_image.img_to_array(img) / 255.0
        except Exception:
            pass
    return X

X_all_raw = load_images_as_array(all_paths, img_size=IMG_SIZE)
y_all_raw = all_labels

# Fase SHUFFLE: seed berbeda dari Grid Search (AUG_SEED) agar seleksi & evaluasi independen
SHUFFLE_SEED = 2025
rng_shuffle = np.random.default_rng(SHUFFLE_SEED)
shuffle_idx = rng_shuffle.permutation(len(X_all_raw))
X_all = X_all_raw[shuffle_idx]
y_all = y_all_raw[shuffle_idx]

# Fase TEST: Stratified 10-Fold CV
K_FOLDS   = 10
CV_EPOCHS = 40
CV_DIR    = os.path.join(OUTPUT_DIR, 'CV_Results')
os.makedirs(CV_DIR, exist_ok=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=False)  # shuffle eksplisit sudah dilakukan di atas

cv_records   = []
cv_histories = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), start=1):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    cw_dict = dict(enumerate(cw))

    tf.keras.backend.clear_session()
    model_cv = build_cnn(dropout_rate=BEST_DR, num_classes=NUM_CLASSES)
    model_cv.compile(
        optimizer=get_optimizer(BEST_OPT, BEST_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    ckpt_cv = os.path.join(CV_DIR, f'fold_{fold_idx:02d}.keras')
    callbacks_cv = [
        keras.callbacks.ModelCheckpoint(filepath=ckpt_cv, monitor='val_loss', save_best_only=True, mode='min', verbose=0),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=0),
    ]

    hist = model_cv.fit(
        X_tr, y_tr,
        epochs=CV_EPOCHS,
        batch_size=BEST_BS,
        validation_data=(X_val, y_val),
        class_weight=cw_dict,
        callbacks=callbacks_cv,
        verbose=0
    )
    cv_histories.append(hist.history)

    val_loss, val_acc = model_cv.evaluate(X_val, y_val, verbose=0)
    y_pred = np.argmax(model_cv.predict(X_val, verbose=0), axis=1)

    report = classification_report(y_val, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

    fold_result = {
        'fold'         : fold_idx,
        'val_loss'     : round(val_loss, 4),
        'val_acc'      : round(val_acc,  4),
        'f1_weighted'  : round(report['weighted avg']['f1-score'],  4),
        'precision_w'  : round(report['weighted avg']['precision'], 4),
        'recall_w'     : round(report['weighted avg']['recall'],    4),
        'f1_no_tumor'  : round(report['no_tumor']['f1-score'],      4),
        'f1_tumor'     : round(report['tumor']['f1-score'],         4),
        'epochs_run'   : len(hist.history['val_loss']),
        'best_val_loss': round(min(hist.history['val_loss']),       4),
    }
    cv_records.append(fold_result)

    print(f"[Fold {fold_idx:02d}/{K_FOLDS}] acc={val_acc*100:.2f}% f1={fold_result['f1_weighted']*100:.2f}%")


df_cv = pd.DataFrame(cv_records).set_index('fold')

metrics_summary = ['val_acc', 'f1_weighted', 'precision_w', 'recall_w', 'f1_no_tumor', 'f1_tumor', 'val_loss']
summary_rows = []
for m in metrics_summary:
    vals = df_cv[m].values
    summary_rows.append({'Metric': m, 'Mean': round(vals.mean(), 4), 'Std': round(vals.std(), 4),
                          'Min': round(vals.min(), 4), 'Max': round(vals.max(), 4)})
df_summary = pd.DataFrame(summary_rows).set_index('Metric')

mean_acc = df_cv['val_acc'].mean()
std_acc  = df_cv['val_acc'].std()
mean_f1  = df_cv['f1_weighted'].mean()
std_f1   = df_cv['f1_weighted'].std()

# Simpan hasil
df_cv.to_csv(os.path.join(CV_DIR, 'cv_fold_results.csv'))
df_summary.to_csv(os.path.join(CV_DIR, 'cv_summary.csv'))
with open(os.path.join(CV_DIR, 'cv_results.json'), 'w') as f:
    json.dump({
        'method'       : 'Select-Shuffle-Test',
        'shuffle_seed' : SHUFFLE_SEED,
        'best_config'  : best_config,
        'fold_results' : cv_records,
        'summary'      : df_summary.reset_index().to_dict(orient='records'),
    }, f, indent=2)

print(f"10-Fold CV selesai. Accuracy={mean_acc*100:.2f}%+/-{std_acc*100:.2f}% F1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%")


# Visualisasi
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle(
    f'Select-Shuffle-Test — 10-Fold CV\nlr={BEST_LR} | bs={BEST_BS} | {BEST_OPT} | dropout={BEST_DR}',
    fontsize=13, fontweight='bold'
)

folds_x = df_cv.index.tolist()

ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(folds_x, df_cv['val_acc'] * 100, color='#3498db', alpha=0.85, edgecolor='white')
ax1.axhline(mean_acc * 100, color='red', linestyle='--', linewidth=1.5, label=f'Mean={mean_acc*100:.2f}%')
ax1.fill_between(folds_x, (mean_acc - std_acc) * 100, (mean_acc + std_acc) * 100,
                 alpha=0.15, color='red', label=f'+/-Std={std_acc*100:.2f}%')
ax1.set_title('Accuracy per Fold', fontweight='bold')
ax1.set_xlabel('Fold')
ax1.set_ylabel('Accuracy (%)')
ax1.set_xticks(folds_x)
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(folds_x, df_cv['f1_weighted'] * 100, color='#2ecc71', alpha=0.85, edgecolor='white')
ax2.axhline(mean_f1 * 100, color='red', linestyle='--', linewidth=1.5, label=f'Mean={mean_f1*100:.2f}%')
ax2.fill_between(folds_x, (mean_f1 - std_f1) * 100, (mean_f1 + std_f1) * 100,
                 alpha=0.15, color='red', label=f'+/-Std={std_f1*100:.2f}%')
ax2.set_title('F1-Score (Weighted) per Fold', fontweight='bold')
ax2.set_xlabel('Fold')
ax2.set_ylabel('F1-Score (%)')
ax2.set_xticks(folds_x)
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
box_data  = [df_cv['val_acc']*100, df_cv['f1_weighted']*100, df_cv['precision_w']*100, df_cv['recall_w']*100]
box_labels = ['Accuracy', 'F1\n(weighted)', 'Precision\n(weighted)', 'Recall\n(weighted)']
bp = ax3.boxplot(box_data, labels=box_labels, patch_artist=True, medianprops=dict(color='red', linewidth=2))
colors_box = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_title('Distribusi Metrik (10 Fold)', fontweight='bold')
ax3.set_ylabel('Score (%)')
ax3.grid(axis='y', alpha=0.3)

min_len = min(len(h['val_loss']) for h in cv_histories)
acc_mat      = np.array([h['accuracy']    [:min_len] for h in cv_histories])
val_acc_mat  = np.array([h['val_accuracy'][:min_len] for h in cv_histories])
loss_mat     = np.array([h['loss']        [:min_len] for h in cv_histories])
val_loss_mat = np.array([h['val_loss']    [:min_len] for h in cv_histories])
ep_x = range(1, min_len + 1)

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(ep_x, acc_mat.mean(axis=0)*100,     color='royalblue', label='Train Acc (mean)')
ax4.plot(ep_x, val_acc_mat.mean(axis=0)*100, color='tomato',    label='Val Acc (mean)',   linestyle='--')
ax4.fill_between(ep_x, (val_acc_mat.mean(0) - val_acc_mat.std(0)) * 100,
                  (val_acc_mat.mean(0) + val_acc_mat.std(0)) * 100,
                  alpha=0.15, color='tomato', label='Val Acc +/-std')
ax4.set_title('Learning Curve Rata-Rata (10 Fold) — Accuracy', fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Accuracy (%)')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(ep_x, loss_mat.mean(axis=0),     color='royalblue', label='Train Loss (mean)')
ax5.plot(ep_x, val_loss_mat.mean(axis=0), color='tomato',    label='Val Loss (mean)',   linestyle='--')
ax5.fill_between(ep_x, val_loss_mat.mean(0) - val_loss_mat.std(0),
                  val_loss_mat.mean(0) + val_loss_mat.std(0),
                  alpha=0.15, color='tomato', label='Val Loss +/-std')
ax5.set_title('Learning Curve Rata-Rata\n(10 Fold) — Loss', fontweight='bold')
ax5.set_xlabel('Epoch')
ax5.set_ylabel('Loss')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

plt.savefig(os.path.join(CV_DIR, 'cv_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Perbandingan — Grid Search: acc={best_config['test_acc']*100:.2f}% f1={best_config['f1_weighted']*100:.2f}% "
      f"| 10-Fold CV: acc={mean_acc*100:.2f}%+/-{std_acc*100:.2f}% f1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%")
